In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from nltk.corpus import wordnet as wn
from tqdm import tqdm
import nltk
import joblib
nltk.download('wordnet')



[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [19]:
df = pd.read_csv("/kaggle/input/nli-training/train.csv")
X_raw = df[["premise", "hypothesis"]]
y = df["label"]
X_train_raw, X_dev_raw, y_train, y_dev = train_test_split(X_raw, y, test_size=0.2, stratify=y)

In [20]:
def get_synsets(word):
    return wn.synsets(word)

def get_antonyms(word):
    antonyms = set()
    for syn in wn.synsets(word):
        for lem in syn.lemmas():
            for ant in lem.antonyms():
                antonyms.add(ant.name())
    return antonyms

# Custom WordNet-based semantic similarity
def semantic_features(X):
    features = []
    for pair in tqdm(X, desc="Extracting semantic features"):
        premise, hypothesis = pair.split(" [SEP] ")
        p_words = set(premise.split())
        h_words = set(hypothesis.split())

        # Synonym overlap
        syn_overlap = sum(
            1 for hw in h_words if get_synsets(hw)
            and any(get_synsets(hw)[0].lowest_common_hypernyms(get_synsets(pw)[0]) for pw in p_words if get_synsets(pw))
        )

        # Antonym detection
        antonyms = sum(
            1 for hw in h_words for pw in p_words
            if get_antonyms(hw) and hw != pw
        )

        features.append([syn_overlap, antonyms])
    return np.array(features)

In [21]:
pipeline = Pipeline([
    ('features', FeatureUnion([
        ('tfidf', Pipeline([
            ('combine', FunctionTransformer(lambda x: x["premise"] + " [SEP] " + x["hypothesis"], validate=False)),
            ('tfidf', TfidfVectorizer(
                lowercase=True,
                max_features=10000,
                ngram_range=(1, 2),
                sublinear_tf=True
            ))
        ])),
        ('semantic', Pipeline([
            ('combine', FunctionTransformer(lambda x: x["premise"] + " [SEP] " + x["hypothesis"], validate=False)),
            ('semantic_feats', FunctionTransformer(semantic_features, validate=False))
        ]))
    ])),
    ('clf', SVC(
        kernel='linear',
        class_weight='balanced',
        C=0.1,
        probability=False,
        verbose=True
    ))
])

In [22]:
pipeline.fit(X_train_raw, y_train)

Extracting semantic features: 100%|██████████| 19545/19545 [01:17<00:00, 252.83it/s]


[LibSVM]...........................................................................................................................................................................................................................................................................................................................................*.............................................................................................................................*......................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('tfidf',
                                                 Pipeline(steps=[('combine',
                                                                  FunctionTransformer(func=<function <lambda> at 0x79ea35873060>)),
                                                                 ('tfidf',
                                                                  TfidfVectorizer(max_features=10000,
                                                                                  ngram_range=(1,
                                                                                               2),
                                                                                  sublinear_tf=True))])),
                                                ('semantic',
                                                 Pipeline(steps=[('combine',
                                                                  FunctionTransformer(func=<function <lambda> at 0x79ea35871a80>)),
                                                                 ('semantic_feats',
                                                                  FunctionTransformer(func=<function semantic_features at 0x79ea35871c60>))]))])),
                ('clf',
                 SVC(C=0.1, class_weight='balanced', kernel='linear',
                     verbose=True))])

In [23]:
y_pred = pipeline.predict(X_dev_raw)

print("\nEvaluation:")
print(classification_report(y_dev, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_dev, y_pred))
print(f"Accuracy: {accuracy_score(y_dev, y_pred):.4f}")
print(f"Macro-F1: {f1_score(y_dev, y_pred, average='macro'):.4f}")
print(f"Precision: {precision_score(y_dev, y_pred, average='macro'):.4f}")
print(f"Recall: {recall_score(y_dev, y_pred, average='macro'):.4f}")
print(f"MCC: {matthews_corrcoef(y_dev, y_pred):.4f}")

Extracting semantic features: 100%|██████████| 4887/4887 [00:19<00:00, 246.93it/s]



Evaluation:
              precision    recall  f1-score   support

           0       0.58      0.66      0.62      2357
           1       0.64      0.56      0.60      2530

    accuracy                           0.61      4887
   macro avg       0.61      0.61      0.61      4887
weighted avg       0.61      0.61      0.61      4887

Confusion Matrix:
 [[1552  805]
 [1104 1426]]
Accuracy: 0.6094
Macro-F1: 0.6091
Precision: 0.6118
Recall: 0.6111
MCC: 0.2228


In [24]:
df_test = pd.read_csv("/kaggle/input/nli-development-dataset/dev.csv")
X_test_raw = df_test[["premise", "hypothesis"]]
test_preds = pipeline.predict(X_test_raw)


Extracting semantic features: 100%|██████████| 6736/6736 [00:26<00:00, 252.89it/s]


In [25]:
joblib.dump(pipeline, 'nli_svm_model.joblib')
output = pd.DataFrame({"prediction": test_preds})
output.to_csv("Group_52_A.csv", index=False)
